In [ ]:
from PyPDF2 import PdfReader 
import glob
import pandas as pd
import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np 
from sklearn.manifold import TSNE
from sklearn import cluster, datasets, mixture
import matplotlib
import cmasher as cmr
from mycolorpy import colorlist as mcp
import re 
from sklearn import cluster, datasets, mixture
from scipy.spatial import ConvexHull,convex_hull_plot_2d
def pdf2list(file):
    reader = PdfReader(file)
    #print(len(reader.pages)) 
      
    # getting a specific page from the pdf file 
    page = reader.pages[0] 
      
    # extracting text from page 
    text = page.extract_text() 
    #print(text)
    if page.extract_text() !='':
        r = re.compile("HC Number*")
        newlist = list(filter(r.match, text.split('\n'))) # Read Note below
        order_number = int(newlist[0][-5::])#(newlist[0].split('#')[-1])
        #order_number = text.split('\n')[2].split('#')[-1]##.find('Sequence')
        r = re.compile("Calculated Molecular Weight*")
        newlist = list(filter(r.match, text.split('\n'))) # Read Note below
        mw = float(newlist[0].split(' ')[-1])
        r = re.compile("Sequence*")
        newlist = list(filter(r.match, text.split('\n'))) # Read Note below
        seq = newlist[0][8:]
        return order_number,mw,seq
    else:
        return np.nan,np.nan,np.nan
l = []
for i in glob.glob('/Users/aelazar/Downloads/Peptide_Synthesis_Service_2023.11.03.09.14.40.223/*_report.stjude.pdf'):
    order_number,mw,seq =  pdf2list(i)
    l.append({'order_number':order_number,'mw':mw,'seq':seq})
orederd_peptides  = pd.DataFrame(l)
orederd_peptides = orederd_peptides[~pd.isna(orederd_peptides.mw)]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from matplotlib.patches import Circle, RegularPolygon
from matplotlib.path import Path
from matplotlib.projections import register_projection
from matplotlib.projections.polar import PolarAxes
from matplotlib.spines import Spine
from matplotlib.transforms import Affine2D


def radar_factory(num_vars, frame='circle'):
    """
    Create a radar chart with `num_vars` Axes.

    This function creates a RadarAxes projection and registers it.

    Parameters
    ----------
    num_vars : int
        Number of variables for radar chart.
    frame : {'circle', 'polygon'}
        Shape of frame surrounding Axes.

    """
    # calculate evenly-spaced axis angles
    theta = np.linspace(0, 2*np.pi, num_vars, endpoint=False)

    class RadarTransform(PolarAxes.PolarTransform):

        def transform_path_non_affine(self, path):
            # Paths with non-unit interpolation steps correspond to gridlines,
            # in which case we force interpolation (to defeat PolarTransform's
            # autoconversion to circular arcs).
            if path._interpolation_steps > 1:
                path = path.interpolated(num_vars)
            return Path(self.transform(path.vertices), path.codes)

    class RadarAxes(PolarAxes):

        name = 'radar'
        PolarTransform = RadarTransform

        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            # rotate plot such that the first axis is at the top
            self.set_theta_zero_location('N')

        def fill(self, *args, closed=True, **kwargs):
            """Override fill so that line is closed by default"""
            return super().fill(closed=closed, *args, **kwargs)

        def plot(self, *args, **kwargs):
            """Override plot so that line is closed by default"""
            lines = super().plot(*args, **kwargs)
            for line in lines:
                self._close_line(line)

        def _close_line(self, line):
            x, y = line.get_data()
            # FIXME: markers at x[0], y[0] get doubled-up
            if x[0] != x[-1]:
                x = np.append(x, x[0])
                y = np.append(y, y[0])
                line.set_data(x, y)

        def set_varlabels(self, labels):
            self.set_thetagrids(np.degrees(theta), labels)

        def _gen_axes_patch(self):
            # The Axes patch must be centered at (0.5, 0.5) and of radius 0.5
            # in axes coordinates.
            if frame == 'circle':
                return Circle((0.5, 0.5), 0.5)
            elif frame == 'polygon':
                return RegularPolygon((0.5, 0.5), num_vars,
                                      radius=.5, edgecolor="k")
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

        def _gen_axes_spines(self):
            if frame == 'circle':
                return super()._gen_axes_spines()
            elif frame == 'polygon':
                # spine_type must be 'left'/'right'/'top'/'bottom'/'circle'.
                spine = Spine(axes=self,
                              spine_type='circle',
                              path=Path.unit_regular_polygon(num_vars))
                # unit_regular_polygon gives a polygon of radius 1 centered at
                # (0, 0) but we want a polygon of radius 0.5 centered at (0.5,
                # 0.5) in axes coordinates.
                spine.set_transform(Affine2D().scale(.5).translate(.5, .5)
                                    + self.transAxes)
                return {'polar': spine}
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

    register_projection(RadarAxes)
    return theta

In [ ]:
import sklearn.preprocessing
import sklearn.pipeline
import scipy.spatial
import numpy as np


def calculate_hull(
        X, 
        scale=1.1, 
        padding="scale", 
        n_interpolate=100, 
        interpolation="quadratic_periodic", 
        return_hull_points=False):
    """
    Calculates a "smooth" hull around given points in `X`.
    The different settings have different drawbacks but the given defaults work reasonably well.
    Parameters
    ----------
    X : np.ndarray
        2d-array with 2 columns and `n` rows
    scale : float, optional
        padding strength, by default 1.1
    padding : str, optional
        padding mode, by default "scale"
    n_interpolate : int, optional
        number of interpolation points, by default 100
    interpolation : str or callable(ix,iy,x), optional
        interpolation mode, by default "quadratic_periodic"

    Inspired by: https://stackoverflow.com/a/17557853/991496
    """
    
    if padding == "scale":

        # scaling based padding
        scaler = sklearn.pipeline.make_pipeline(
            sklearn.preprocessing.StandardScaler(with_std=False),
            sklearn.preprocessing.MinMaxScaler(feature_range=(-1,1)))
        points_scaled = scaler.fit_transform(X) * scale
        hull_scaled = scipy.spatial.ConvexHull(points_scaled, incremental=True)
        hull_points_scaled = points_scaled[hull_scaled.vertices]
        hull_points = scaler.inverse_transform(hull_points_scaled)
        hull_points = np.concatenate([hull_points, hull_points[:1]])
    
    elif padding == "extend" or isinstance(padding, (float, int)):
        # extension based padding
        # TODO: remove?
        if padding == "extend":
            add = (scale - 1) * np.max([
                X[:,0].max() - X[:,0].min(), 
                X[:,1].max() - X[:,1].min()])
        else:
            add = padding
        points_added = np.concatenate([
            X + [0,add], 
            X - [0,add], 
            X + [add, 0], 
            X - [add, 0]])
        hull = scipy.spatial.ConvexHull(points_added)
        hull_points = points_added[hull.vertices]
        hull_points = np.concatenate([hull_points, hull_points[:1]])
    else:
        raise ValueError(f"Unknown padding mode: {padding}")
    
    # number of interpolated points
    nt = np.linspace(0, 1, n_interpolate)
    
    x, y = hull_points[:,0], hull_points[:,1]
    
    # ensures the same spacing of points between all hull points
    t = np.zeros(x.shape)
    t[1:] = np.sqrt((x[1:] - x[:-1])**2 + (y[1:] - y[:-1])**2)
    t = np.cumsum(t)
    t /= t[-1]

    # interpolation types
    if interpolation is None or interpolation == "linear":
        x2 = scipy.interpolate.interp1d(t, x, kind="linear")(nt)
        y2 = scipy.interpolate.interp1d(t, y, kind="linear")(nt)
    elif interpolation == "quadratic":
        x2 = scipy.interpolate.interp1d(t, x, kind="quadratic")(nt)
        y2 = scipy.interpolate.interp1d(t, y, kind="quadratic")(nt)

    elif interpolation == "quadratic_periodic":
        x2 = scipy.interpolate.splev(nt, scipy.interpolate.splrep(t, x, per=True, k=4))
        y2 = scipy.interpolate.splev(nt, scipy.interpolate.splrep(t, y, per=True, k=4))
    
    elif interpolation == "cubic":
        x2 = scipy.interpolate.CubicSpline(t, x, bc_type="periodic")(nt)
        y2 = scipy.interpolate.CubicSpline(t, y, bc_type="periodic")(nt)
    else:
        x2 = interpolation(t, x, nt)
        y2 = interpolation(t, y, nt)
    
    X_hull = np.concatenate([x2.reshape(-1,1), y2.reshape(-1,1)], axis=1)
    if return_hull_points:
        return X_hull, hull_points
    else:
        return X_hull


def draw_hull(
        X, 
        scale=1.1, 
        padding="scale", 
        n_interpolate=100, 
        interpolation="quadratic_periodic",
        plot_kwargs=None, 
        ax=None):
    """Uses `calculate_hull` to draw a hull around given points.

    Parameters
    ----------
    X : np.ndarray
        2d-array with 2 columns and `n` rows
    scale : float, optional
        padding strength, by default 1.1
    padding : str, optional
        padding mode, by default "scale"
    n_interpolate : int, optional
        number of interpolation points, by default 100
    interpolation : str or callable(ix,iy,x), optional
        interpolation mode, by default "quadratic_periodic"
    plot_kwargs : dict, optional
        `matplotlib.pyplot.plot` kwargs, by default None
    ax : `matplotlib.axes.Axes`, optional
        [description], by default None
    """

    if plot_kwargs is None:
        plot_kwargs = {}

    X_hull = calculate_hull(
        X, scale=scale, padding=padding, n_interpolate=n_interpolate, interpolation=interpolation)
    if ax is None:
        ax= plt.gca()
    plt.plot(X_hull[:,0], X_hull[:,1], **plot_kwargs)


def draw_rounded_hull(X, padding=0.1, line_kwargs=None, ax=None):
    """Plots a convex hull around points with rounded corners and a given padding.

    Parameters
    ----------
    X : np.array
        2d array with two columns and n rows
    padding : float, optional
        padding between hull and points, by default 0.1
    line_kwargs : dict, optional
        line kwargs (used for `matplotlib.pyplot.plot` and `matplotlib.patches.Arc`), by default None
    ax : matplotlib.axes.Axes, optional
        axes to plat on, by default None
    """

    default_line_kwargs = dict(
        color="black",
        linewidth=1
    )
    if line_kwargs is None:
        line_kwargs = default_line_kwargs
    else:
        line_kwargs = {**default_line_kwargs, **line_kwargs}

    if ax is None:
        ax = plt.gca()

    hull = scipy.spatial.ConvexHull(X)
    hull_points = X[hull.vertices]

    hull_points = np.concatenate([hull_points[[-1]], hull_points, hull_points[[0]]])

    diameter = padding * 2
    for i in range(1, hull_points.shape[0] - 1):

        # line
        
        # source: https://stackoverflow.com/a/1243676/991496
        
        norm_next = np.flip(hull_points[i] - hull_points[i + 1]) * [-1, 1]
        norm_next /= np.linalg.norm(norm_next)

        norm_prev = np.flip(hull_points[i - 1] - hull_points[i]) * [-1, 1]
        norm_prev /= np.linalg.norm(norm_prev)

        # plot line
        line = hull_points[i:i+2] + norm_next * diameter / 2
        ax.plot(line[:,0], line[:,1], **line_kwargs) 

        # arc

        angle_next = np.rad2deg(np.arccos(np.dot(norm_next, [1,0])))
        if norm_next[1] < 0:
            angle_next = 360 - angle_next

        angle_prev = np.rad2deg(np.arccos(np.dot(norm_prev, [1,0])))
        if norm_prev[1] < 0:
            angle_prev = 360 - angle_prev

        arc = patches.Arc(
            hull_points[i], 
            diameter, diameter,
            angle=0, fill=False, theta1=angle_prev, theta2=angle_next,
            **line_kwargs)

        ax.add_patch(arc)

In [ ]:
orederd_peptides[orederd_peptides.seq.str.find('YEVHH')!=-1]

In [ ]:
hTMs_uniprot = pd.read_csv('/Users/aelazar/Downloads/idmapping_2023_10_27.tsv',delimiter='\t')


In [ ]:
# #c  = 0 
# ind  = []
# seq_in = []
# for i in orederd_peptides.seq:
#     for j in df_membrane.index:
#         if levenshtein_distance(df_membrane.loc[j,'AAseq'],i)==0:
#         #if len(df_membrane[df_membrane.AAseq.str.find(i[1:-1])!=-1])==1:
#             ind.append(j)
#             seq_in.append(i)
# ind.append(571)#APP
# ind.append(1188)#AGVVTTVMG,P08138
# ind.append(1056)#TNFRSF13C,DKVIILS

In [ ]:
path = "/Users/aelazar/Library/CloudStorage/OneDrive-St.JudeChildren'sResearchHospital/Assaf-Vikas/IDR_dSAMs/ThT/"
plate_pos2seq = pd.read_excel('%s/2023_11_08_Peptide_Dictionary.xlsx'%path,sheet_name='Sheet2',index_col=0)
l =[]
for ind in plate_pos2seq.index:
    for j in plate_pos2seq.columns:
        if j<10:
            if pd.isna(plate_pos2seq.loc[ind,j])==False:
                l.append({'well':'%s0%d'%(ind,j),'odder_number':int(plate_pos2seq.loc[ind,j])})     
            else:
                print()
        if j>=10:
            if pd.isna(plate_pos2seq.loc[ind,j])==False:
                l.append({'well':'%s%d'%(ind,j),'odder_number':int(plate_pos2seq.loc[ind,j])})     
plate_pos2seq_df = pd.DataFrame(l)

In [ ]:
df_membrane = pd.read_pickle('HTS/umap.pkl')
for i in df_membrane.index:
    df_membrane.loc[i,'gene_name'] = hTMs_uniprot.loc[hTMs_uniprot[hTMs_uniprot.Entry==df_membrane.loc[i,'acc']].index]['Gene Names'].values[0].split(' ')[0]
gmm = mixture.GaussianMixture(n_components=3, covariance_type="full",random_state=42)
gmm.fit(df_membrane.loc[:,['x','y']])

for i in enumerate(gmm.predict(df_membrane.loc[:,['x','y']])):
    df_membrane.at[df_membrane.index[i[0]],'gmm_dSAMs'] = i[1]

In [ ]:
df_membrane.gmm_dSAMs.value_counts()

#['#414042','#FFBF00','#697793']

In [ ]:
l = []
reps = glob.glob('%s/2023_11_08_Peptide_R*.xlsx'%path)
for rep in reps:
    r = int(rep.split('/')[-1].split('_')[-1].split('.')[0][1:])
    dff = pd.read_excel(rep,header=10)
    for ind in dff.index:
        well = dff.loc[ind,'Unnamed: 0']
        order_number = plate_pos2seq_df[plate_pos2seq_df.well==well].odder_number.values[0]
        seq = orederd_peptides[orederd_peptides.order_number==order_number].seq
        #print(seq)
        if seq.empty==False:
            seq_ = seq.values[0]
            for col in dff.columns[2::]:
                l.append({'well':well,'nm':col,'ThT':dff.loc[ind,col],'rep':r,'Seq':seq_})
        else:
            for col in dff.columns[2::]:
                l.append({'well':well,'nm':col,'ThT':dff.loc[ind,col],'rep':r,'Seq':'AA'})
            
df = pd.DataFrame(l)
inds = df[df.Seq=='GYEVHHQKLVFFAED'].index
df.loc[inds,'Seq'] = 'YEVHHQKLVFFA'#GYEVHHQKLVFFAED
inds = df[df.Seq=='AGVVTTVMG'].index
df.loc[inds,'Seq'] = 'AGVVTTVMGS'
inds = df[df.Seq=='DKVIILS'].index
df.loc[inds,'Seq'] = 'DKVIILSP'
well_order= df[df.nm==490].loc[:,['Seq','ThT']].groupby(['Seq']).mean().reset_index().sort_values(by='ThT')
#df[df.nm==490].groupby(['Seq']).mean().reset_index().sort_values(by='ThT')
inds = df[df.Seq=='VQIVYI'].index
df.loc[inds,'Seq'] = 'VQIVYI'
for i in [0,1,2,3]:
    aa = df_membrane[(df_membrane.AAseq.isin(df.Seq))&(df_membrane.gmm_dSAMs.isin([i]))].AAseq
    df.loc[df[df.Seq.isin(aa)].index,'gmm']=i
df.loc[df[df.well=='H10'].index,'gmm']=5# APP 2M 
df.loc[df[df.well=='C03'].index,'gmm']=6 #Pos_control

for i in df[(df.gmm.isin(range(4)))].index:
    seq = df.loc[i,'Seq']
    gene_name = df_membrane[df_membrane.AAseq.str.find(seq)!=-1].gene_name.values[0]
    df.loc[i,'gene_name'] = gene_name
for i in df[df.gmm==5].index:
    df.loc[i,'gene_name'] = 'APP**'
for i in df[df.gmm==6].index:
    df.loc[i,'gene_name'] = 'MAPT'

In [ ]:
print(df[df.Seq.str.find('VPHQK')!=-1].iloc[0])
print(plate_pos2seq_df[plate_pos2seq_df.well=='H10'])


In [ ]:
df_membrane[(df_membrane.AAseq.isin(df[~pd.isna(df.gene_name)].Seq))&(df_membrane.gmm_dSAMs==2)].AAseq.to_clipboard(index=0)

In [ ]:
ind  = df_membrane[df_membrane.AAseq.isin(df[~pd.isna(df.gene_name)].Seq)].index
fig, ax = plt.subplots(figsize=(3, 3.5))
#colors_gmm = ['#EAA764','#009FB7','#003844']#['#20321F','#497147','#84B082']
colors_gmm = ['#414042','#FFBF00','#697793']
ax.grid()
for i,j in zip([0,1,2], colors_gmm):#,['#926189','#B2493E','#4C6CAB']
    #hull = ConvexHull(df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy())
    points =df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy()
    draw_hull(points,scale=1.05,plot_kwargs={'color':j,'lw':3})
    # for simplex in hull.simplices:
    #     ax.plot(points[simplex, 0], points[simplex, 1], 'k',zorder=3)



#sns.scatterplot(x='x',y='y',data=df_membrane,hue='gmm_dSAMs',edgecolor=None,s=10,alpha=1,palette='Dark2',zorder=2,ax=ax)
sns.scatterplot(x='x',y='y',data=df_membrane,edgecolor=None,s=10,alpha=1,zorder=2,ax=ax,color='#B4B5B4')

sns.scatterplot(x='x',y='y',data=df_membrane.loc[ind],edgecolor='k',hue='gmm_dSAMs',s=35,alpha=1,palette=['#B4B5B4','#B4B5B4','#B4B5B4'],zorder=2)#palette= ['#926189','#B2493E','#4C6CAB']



#plt.legend([],frameon=False)#C03830
plt.locator_params(nbins=4)
plt.legend(ncol=4)
plt.axis('off')
plt.xlabel('t-SNE',fontsize=12)
plt.ylabel('t-SNE',fontsize=12)
plt.legend([],frameon=False)
plt.savefig('hull_test.svg',bbox_inches='tight',transparent=True)
#plt.savefig('hull_gmm.svg',bbox_inches='tight',transparent=True)

In [ ]:
sns.scatterplot(x='x',y='y',data=df_membrane,edgecolor=None,s=10,alpha=1,zorder=2,color='#B4B5B4')

sns.scatterplot(x='x',y='y',data=df_membrane.loc[ind],edgecolor='k',hue='gmm_dSAMs',s=35,alpha=1,zorder=2)#palette= ['#926189','#B2493E','#4C6CAB']


In [ ]:
df_membrane.loc[ind][df_membrane.gmm_dSAMs==2].AAseq

In [ ]:
df_membrane.loc[ind].gmm_dSAMs.value_counts()

In [ ]:
ternary_df = pd.read_pickle('ternary_df.pkl')

In [ ]:
import matplotlib
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#CCCCCC","#EEA155"])###3C4676


In [ ]:
ternary_df

In [ ]:
test = set.intersection(set(df_membrane[df_membrane.gmm_dSAMs>0].index),set(ind))
no_test = set.difference(set(df_membrane[df_membrane.gmm_dSAMs>=0].index),test)
df_membrane[(df_membrane.gmm_dSAMs==0)&(df_membrane.index.isin(ind))]

In [ ]:
df_membrane['pos'] = df_membrane.AAseq.map(dict(zip(ternary_df.AAseq,ternary_df.pos)))
df_membrane['neg'] = df_membrane.AAseq.map(dict(zip(ternary_df.AAseq,ternary_df.neg)))
df_membrane['hydro'] = df_membrane.AAseq.map(dict(zip(ternary_df.AAseq,ternary_df.hydro)))
df_membrane['cluster'] = ''
df_membrane.loc[df_membrane[df_membrane.gmm_dSAMs==0].index,['cluster']] = colors_gmm[0]#'#926189'
df_membrane.loc[df_membrane[df_membrane.gmm_dSAMs==1].index,['cluster']] = colors_gmm[1]#'#B2493E'
df_membrane.loc[df_membrane[df_membrane.gmm_dSAMs==2].index,['cluster']] = colors_gmm[2]#'#4C6CAB'
df_membrane['tested'] = ''
df_membrane.loc[list(test),'tested'] = 'k'
df_membrane.loc[list(no_test),'tested'] = 'white'


In [ ]:
df_membrane.cluster.value_counts()

In [ ]:
g = sns.clustermap(df_membrane.loc[df_membrane[df_membrane.gmm_dSAMs<3].index,['pos','neg','hydro']].transpose(),cmap=ww,metric='cosine',
               col_cluster=1,row_cluster=0,method='single',col_colors=[df_membrane.cluster],figsize=(10,3),dendrogram_ratio=.1,colors_ratio=0.1)
g.ax_heatmap.set_xticklabels([])

In [ ]:
df_membrane['color'] = ''
df_membrane.loc[df_membrane[df_membrane.gmm_dSAMs==0].index,['color']] = 'r'
df_membrane.loc[df_membrane[df_membrane.gmm_dSAMs==1].index,['color']] = 'g'
df_membrane.loc[df_membrane[df_membrane.gmm_dSAMs==2].index,['color']] = 'b'


In [ ]:
import ternary
fig, ax = plt.subplots(figsize=(5,5)) 
scale = 100
figure, tax = ternary.figure(scale=scale,ax=ax)
tax.boundary(linewidth=2.0,zorder=0)
tax.gridlines(multiple=10, color="gray")
#tax.scatter(l, color='gray',alpha=.3,s=40)
tax.scatter(df_membrane.loc[df_membrane[df_membrane.gmm_dSAMs==0].index,['pos','neg','hydro']].to_numpy(), color=colors_gmm[0],s=100,alpha=.5)
tax.scatter(df_membrane.loc[df_membrane[df_membrane.gmm_dSAMs==1].index,['pos','neg','hydro']].to_numpy(), color=colors_gmm[1],s=100,alpha=.51)
tax.scatter(df_membrane.loc[df_membrane[df_membrane.gmm_dSAMs==2].index,['pos','neg','hydro']].to_numpy(), color=colors_gmm[2],s=100,alpha=.51)
tax.scatter(df_membrane.loc[df_membrane[(df_membrane.gmm_dSAMs==0)&(df_membrane.index.isin(ind))].index,['pos','neg','hydro']].to_numpy(), color=colors_gmm[0]
            ,s=100,alpha=.51,edgecolor='k')
tax.scatter(df_membrane.loc[df_membrane[(df_membrane.gmm_dSAMs==1)&(df_membrane.index.isin(ind))].index,['pos','neg','hydro']].to_numpy(), color=colors_gmm[1]
            ,s=100,alpha=.51,edgecolor='k')
tax.scatter(df_membrane.loc[df_membrane[(df_membrane.gmm_dSAMs==2)&(df_membrane.index.isin(ind))].index,['pos','neg','hydro']].to_numpy(), color=colors_gmm[2]
            ,s=100,alpha=.51,edgecolor='k')
# tax.scatter(l3, color='#B2493E',s=100,alpha=.5)
# tax.scatter(l4, color='#4C6CAB',s=100,alpha=.5)
# tax.scatter([(1,0,0)] color='b',alpha=1,s=100)
# tax.scatter([(0,1,0)], color='red',alpha=1,s=100)
# tax.scatter([(0,0,1)], color='orange',alpha=1,s=100)
#tax.scatter([(30,60,10)], color='orange',alpha=1,s=100)
tax.ticks(axis='lbr', linewidth=0, multiple=20,fontsize=16,offset=0.02)
# tax.left_axis_label("¬ Charged", fontsize=20,offset=0.15)
# tax.right_axis_label("Positive", fontsize=20,offset=0.15)
# tax.bottom_axis_label("Negative", fontsize=20,offset=0.15)
plt.axis('off')
#tax.show()
tax.savefig('tax_gmm.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(5,5)) 
scale = 100
figure, tax = ternary.figure(scale=scale,ax=ax)
tax.boundary(linewidth=2.0,zorder=0)
tax.gridlines(multiple=10, color="gray")
#tax.scatter(l, color='gray',alpha=.3,s=40)
tax.scatter(df_membrane.loc[df_membrane[(df_membrane.gmm_dSAMs==1)&(df_membrane.index.isin(ind))].index,['pos','neg','hydro']].to_numpy(), color=colors_gmm[1]
            ,s=150,alpha=.8,edgecolor=None)
tax.ticks(axis='lbr', linewidth=0, multiple=20,fontsize=16,offset=0.02)
# tax.left_axis_label("¬ Charged", fontsize=20,offset=0.15)
# tax.right_axis_label("Positive", fontsize=20,offset=0.15)
# tax.bottom_axis_label("Negative", fontsize=20,offset=0.15)
plt.axis('off')
tax.savefig('tax_gmm_1.svg',bbox_inches='tight',transparent=True)

In [ ]:
df['log_ThT']=np.log10(df['ThT'])

In [ ]:
for i in df[(df.nm==490)&(df.gmm.isin(range(7)))].index:
    seq = df.loc[i,'Seq']
    gene = df.loc[i,'gene_name']
    df.loc[i,'comb'] = '%s %s'%(seq,gene)

In [ ]:
df[(df.nm==490)&(df.gmm.isin(range(7)))].index

In [ ]:
df.loc[df[(df.nm==490)&(df.gmm.isin(range(7)))].index,['well','comb','gmm']].groupby(['well','comb','gmm']).mean()

In [ ]:
ord = df.loc[df[(df.nm==490)&(df.gmm.isin(range(7)))].index,['well','comb','gmm']].groupby(['well','comb','gmm']).mean().reset_index().sort_values(by='gmm')['comb']

In [ ]:
ThT_tested = pd.read_pickle('ternary_ThT_tested.pkl')

In [ ]:
]

In [ ]:
df.loc[df[df.Seq.isin(ThT_tested[ThT_tested.cluster==0].AAseq)].index,'gmm_new'] = 0 
df.loc[df[df.Seq.isin(ThT_tested[ThT_tested.cluster==1].AAseq)].index,'gmm_new'] = 2
df.loc[df[df.Seq.isin(ThT_tested[ThT_tested.cluster==2].AAseq)].index,'gmm_new'] = 1
# df.loc[df[df.Seq.isin(ThT_tested[(ThT_tested.neg.between(0,50))&(ThT_tested.hydro<100)&(ThT_tested.pos<20)].AAseq)].index,'gmm_new'] = 1 


# df.loc[df[df.Seq.isin(ThT_tested[(ThT_tested.pos.between(0,50))&(ThT_tested.hydro<100)&(ThT_tested.neg<10)].AAseq)].index,'gmm_new'] =2 


In [ ]:
df_membrane[df_membrane.AAseq.isin(set(df[(df.gmm_new==1)].Seq))]
df

In [ ]:
fig, ax = plt.subplots(figsize=(6, 15))
ord_v2 = df.loc[df[(df.nm==490)&(df.gmm.isin(range(7)))].index,['well','comb','gmm_new']].groupby(['well','comb']).mean().reset_index().sort_values(by='gmm_new')['comb']
#ax.grid(zorder=1)
#sns.scatterplot(y='Seq',x='ThT',data=df[(df.nm==490)&(df.gmm.isin(range(7)))],order=ord,
#                color='k',edgecolor=None,s=5,zorder=25)
sns.barplot(y='comb',x='ThT',data=df[(df.nm==490)&(df.gmm.isin(range(7)))],width=.96,
            order=ord_v2,errorbar=('se'),palette=['#674CAC','#BB1919','#3C4776'],hue='gmm_new',zorder=2,dodge=False)
plt.xscale('log')
plt.yticks(rotation=0)
plt.tight_layout()
plt.axvline(df[(df.well=='H12')&(df.nm==490)].ThT.mean(),color='k',ls='--',lw=.5)#APP 2M 
#ax.grid(zorder=0)
# genes = []
# for seq in enumerate(ord):
#     gene_name = df[df.Seq==seq[1]].gene_name.values[0]
#     ax.text(300,(seq[0]/1)+0.2,'%s'%gene_name,fontsize=16)
#     genes.append(gene_name)
#ax2= ax.twiny()
plt.xticks(fontsize=16)
#ax2.set_yticklabels(genes)
plt.ylabel('')
plt.xlabel('ThT fluorescence intensity\n(490nm)',fontsize=16)
plt.yticks(fontsize=16)
plt.savefig('plate_ThT_test_bar.svg',bbox_inches='tight',transparent=False)

In [ ]:
df[~df.well.isin(['H06','H07'])]# duplicate MET and MET_py


In [ ]:
dff = df[~df.well.isin(['H06','H07'])]
i =1 

In [ ]:
dff.loc[dff[(dff.nm==490)&(dff.gmm_new==i)].index,['ThT','comb']]

In [ ]:
#dff = df[~df.well.isin(['H06','H07'])]
fig = plt.figure(figsize = (12,4),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=3,gridspec_kw = {'hspace':.01,'wspace':.05})
palette=['#674CAC','#BB1919','#3C4776']
l =[]
for i in range(3):
    
    ord = dff.loc[dff[(dff.nm==490)&(dff.gmm_new==i)].index,['ThT','comb']].groupby(['comb']).mean().reset_index().sort_values(by='ThT')['comb']
    l.append(dff.loc[dff[(dff.nm==490)&(dff.gmm_new==i)].index,['ThT','comb']].groupby(['comb']).mean().reset_index().sort_values(by='ThT')['comb'].to_list())
    sns.pointplot(x='comb',y='ThT',data=dff[(dff.nm==490)&(dff.gmm_new==i)],order=ord,errorbar=('se'),zorder=3,ax=axs[i],color=palette[i],linestyles='none')#,width=.96
    axs[i].set_yscale('log')
    axs[i].set_xticklabels(axs[i].get_xticklabels(), rotation=90,fontsize=13)
    axs[i].set_ylim([100,5e4])
    genes = [axs[i].get_xticklabels()[j].get_text().split(' ')[-1] for j in range(len(ord))]
    seqs = [axs[i].get_xticklabels()[j].get_text().split(' ')[0] for j in range(len(ord))]
    # axs[i].axhline(dff[(dff.well=='C03')&(dff.nm==490)].ThT.mean(),color='#3C4776',ls='--',lw=2,zorder=0)#+ 
    # axs[i].axhline(dff[(dff.well=='D12')&(dff.nm==490)].ThT.mean(),color='#CC6159',ls='--',lw=2,zorder=0)#HTR7 Mut 
    axs[i].fill_between([-.5,len(ord_v2)], 0,200 
               ,color='#EC3737', alpha=.2,lw=0)
    axs[i].fill_between([-.5,len(ord)], 200,dff[(dff.well=='C03')&(dff.nm==490)].ThT.mean() 
           ,color='#CCCCCC', alpha=.51,lw=0)
    axs[i].fill_between([-.5,len(ord)], dff[(dff.well=='C03')&(dff.nm==490)].ThT.mean() ,5e4
           ,color='#B8B8B8', alpha=.51,lw=0)
    axs[i].set_xlim([-.50,len(ord)])
    axs[i].set_xticklabels(seqs,fontsize=12)
    #axs[i].set_yticklabels(axs[0].get_xticklabels(), rotation=0,fontsize=14)

    for h in enumerate(genes):
       axs[i].text(h[0],1e3,'%s'%h[1],rotation=90,fontsize=12) 
    if i in [1,2]:
        axs[i].set_ylabel('')
        axs[i].set_yticklabels([])
        axs[i].set_xlabel('')
       
axs[0].set_ylabel('ThT fluorescence intensity\n(490nm)',fontsize=14)
axs[0].set_xlabel('')
axs[0].set_yticklabels(axs[0].get_yticklabels(), rotation=0,fontsize=16)

#plt.savefig('plate_ThT_test.svg',bbox_inches='tight',transparent=False)

In [ ]:
ll = [item for sublist in l for item in sublist]
fig, ax = plt.subplots(figsize=(12, 6))
palette= ['#926189','#B2493E','#4C6CAB']
#sns.pointplot(x='comb',y='ThT',data=dff[(dff.nm==490)],order=ll,errorbar=('se'),zorder=3,linestyles='none',hue='gmm',palette=palette,native_scale=True)#
sns.barplot(x='comb',y='ThT',data=dff[(dff.nm==490)],order=ll,zorder=2,hue='gmm_new',palette=palette,native_scale=True,errorbar=('se'),lw=0,width=.95,
           dodge=False)#
ax.set_yscale('log')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90,fontsize=13)
ax.set_ylim([100,5e4])
ax.fill_between([-.5,len(ll)], 0,200 
           ,color='#C03830', alpha=.6,lw=0)
ax.fill_between([-.5,len(ll)], 200,dff[(dff.well=='C03')&(dff.nm==490)].ThT.mean() 
       ,color='#E3E3E3', alpha=.6,lw=0)
ax.fill_between([-.5,len(ll)], dff[(dff.well=='C03')&(dff.nm==490)].ThT.mean() ,5e4
       ,color='#898989', alpha=.6,lw=0)
ax.set_xlim([-.50,len(ll)])
ax.set_ylabel('ThT fluorescence intensity\n(490nm)',fontsize=16)
plt.tight_layout()
ax.axvline(10.5,lw=1,c='k')
ax.axvline(25.5,lw=1,c='k')
plt.legend([],frameon=False)
ax.set_xlabel('')
plt.savefig('plate_ThT_test_V3.svg',bbox_inches='tight',transparent=False)

In [ ]:
len(ll)

In [ ]:
dff[(dff.nm==490)].to_pickle('ThT_96.pkl')
ThT_96 = dff[(dff.nm==490)]

In [ ]:
ThT_96 = pd.read_pickle('ThT_96.pkl')
#ThT_96[(ThT_96.nm==490)&(ThT_96.Seq.str.find('VSPWVLQS')]

In [ ]:
dff = ThT_96

In [ ]:
df.drop_duplicates()

In [ ]:
cut = df[(df.well=='H12')&(df.nm==490)].ThT.mean()+df[(df.well=='H12')&(df.nm==490)].ThT.std()

In [ ]:

#.groupby(['Seq','gene_name','comb']).mean().reset_index()

In [ ]:
for i,j in zip([0,1,2],['#705A8F','#B93E37','#043278']):
    N = len(ThT_96.loc[ThT_96[ThT_96.gmm_new==i].index,['gene_name','comb']].drop_duplicates())#len(ThT_96.loc[ThT_96[(ThT_96.gmm_new==i)].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name']).mean().reset_index()['ThT'].values)
    temp = ThT_96.loc[ThT_96[(ThT_96.gmm_new==i)].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name','comb']).mean().reset_index()#ThT_96.loc[ThT_96[(ThT_96.gmm_new==i)].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name','comb']).mean().reset_index()
    temp['sem'] = ThT_96.loc[ThT_96[(ThT_96.gmm_new==i)].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name','comb']).sem().reset_index()['ThT']#ThT_96.loc[ThT_96[(ThT_96.gmm_new==i)].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name','comb'])
    temp['ThT_Plus_sem'] = temp['ThT'] + temp['sem']
    temp['ThT_minus_sem'] = temp['ThT'] - temp['sem']
    temp=temp.sort_values(by='ThT')
    print(len(temp))
    print(len(temp[temp.ThT>cut]))
#    m = ThT_96[(ThT_96.gmm_new==i)].groupby(['Seq','gene_name','comb']).mean().reset_index().sort_values(by='ThT')['ThT'] - ThT_96[(ThT_96.gmm_new==i)].groupby(['Seq','gene_name','comb']).mean().reset_index().sort_values(by='ThT')['ThT']
#    p = ThT_96[(ThT_96.gmm_new==i)].groupby(['Seq','gene_name','comb']).mean().reset_index().sort_values(by='ThT')['ThT'] + ThT_96[(ThT_96.gmm_new==i)].groupby(['Seq','gene_name','comb']).se().reset_index().sort_values(by='ThT')['ThT']
     
    theta = radar_factory(N, frame='polygon')
    fig, ax = plt.subplots(figsize=(5,5),subplot_kw=dict(projection='radar'))
    ax.plot(theta, temp['ThT'], color=j,lw=2,zorder=0)
    ax.plot(theta,temp['ThT_Plus_sem'].values, color= j,alpha=0.5)
    ax.plot(theta,temp['ThT_minus_sem'].values, color= j,alpha=0.5)
    # ax.plot(theta,p['ThT'].values, color='#8C6785',alpha=0.5)
    for k in temp.index:
        temp.loc[k,'comb_v2'] = temp.loc[k,'comb'].replace(' ','$\n$')
    ax.set_yscale('log')
    ax.set_ylim([10,5e4])
    ax.plot(np.linspace(0, 2*np.pi, 100), [200] * len(np.linspace(0, 2*np.pi, 100)), c='#D98883',zorder=0,lw=2,ls='--')
    ax.plot(np.linspace(0, 2*np.pi, 100), [901.3333333333334] * len(np.linspace(0, 2*np.pi, 100)), c='#B8B8B8',lw=2,ls='--',zorder=0)
    ax.set_varlabels(temp['comb_v2'].values)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=16)
    plt.tight_layout()
    #plt.savefig('ThT_gmm_%d_ploar_test.svg'%i,bbox_inches='tight',transparent=False)

In [ ]:
ThT_96.loc[ThT_96[ThT_96.Seq.isin(df_membrane.loc[ind][df_membrane.gmm_dSAMs==1].AAseq)].index,['gene_name','comb']]

In [ ]:
fig, ax = plt.subplots(figsize=(2,2))
plt.text(.5,.5,'$amyloid$')
plt.savefig('APP_text.svg')

In [ ]:
for i,j in zip([0,1,2],['#414042','#96615D','#697793']):
    N = len(ThT_96.loc[ThT_96[ThT_96.Seq.isin(df_membrane.loc[ind][df_membrane.gmm_dSAMs==i].AAseq)].index,['gene_name','comb']].drop_duplicates())#len(ThT_96.loc[ThT_96[(ThT_96.gmm_new==i)].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name']).mean().reset_index()['ThT'].values)
    temp = ThT_96.loc[ThT_96[ThT_96.Seq.isin(df_membrane.loc[ind][df_membrane.gmm_dSAMs==i].AAseq)].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name','comb']).mean().reset_index()#ThT_96.loc[ThT_96[(ThT_96.gmm_new==i)].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name','comb']).mean().reset_index()
    temp['sem'] = ThT_96.loc[ThT_96[ThT_96.Seq.isin(df_membrane.loc[ind][df_membrane.gmm_dSAMs==i].AAseq)].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name','comb']).sem().reset_index()['ThT']#ThT_96.loc[ThT_96[(ThT_96.gmm_new==i)].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name','comb'])
    temp['ThT_Plus_sem'] = temp['ThT'] + temp['sem']
    temp['ThT_minus_sem'] = temp['ThT'] - temp['sem']
    temp=temp.sort_values(by='ThT')
    print(len(temp))
    #print(len(temp[temp.ThT>cut]))
#    m = ThT_96[(ThT_96.gmm_new==i)].groupby(['Seq','gene_name','comb']).mean().reset_index().sort_values(by='ThT')['ThT'] - ThT_96[(ThT_96.gmm_new==i)].groupby(['Seq','gene_name','comb']).mean().reset_index().sort_values(by='ThT')['ThT']
#    p = ThT_96[(ThT_96.gmm_new==i)].groupby(['Seq','gene_name','comb']).mean().reset_index().sort_values(by='ThT')['ThT'] + ThT_96[(ThT_96.gmm_new==i)].groupby(['Seq','gene_name','comb']).se().reset_index().sort_values(by='ThT')['ThT']
     
    theta = radar_factory(N, frame='polygon')
    fig, ax = plt.subplots(figsize=(5,5),subplot_kw=dict(projection='radar'))
    ax.plot(theta, temp['ThT'], color=j,lw=2,zorder=0)
    ax.plot(theta,temp['ThT_Plus_sem'].values, color= j,alpha=0.5)
    ax.plot(theta,temp['ThT_minus_sem'].values, color= j,alpha=0.5)
    # ax.plot(theta,p['ThT'].values, color='#8C6785',alpha=0.5)
    for k in temp.index:
        temp.loc[k,'comb_v2'] = temp.loc[k,'comb'].replace(' ','$\n$')
    ax.set_yscale('log')
    ax.set_ylim([10,5e4])
    ax.plot(np.linspace(0, 2*np.pi, 100), [cut] * len(np.linspace(0, 2*np.pi, 100)), c='#D98883',zorder=0,lw=2,ls='--')
    ax.plot(np.linspace(0, 2*np.pi, 100), [901.3333333333334] * len(np.linspace(0, 2*np.pi, 100)), c='#00ffff',lw=2,ls='--',zorder=0)
    ax.set_varlabels(temp['comb_v2'].values)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=16)
    plt.tight_layout()
    plt.savefig('ThT_gmm_%d_ploar_test_v2.svg'%i,bbox_inches='tight',transparent=False)

In [ ]:
fig, ax = plt.subplots(figsize=(2, 3))
import random

    
d = ThT_96.loc[ThT_96[ThT_96.gmm_new.isin([0])].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name','comb']).mean().reset_index()
v = ThT_96.loc[ThT_96[ThT_96.gmm_new.isin([0])].index,['Seq','gene_name','comb','ThT']].groupby(['Seq','gene_name','comb']).sem().reset_index()
#plt.scatter([1 for i in d.ThT], d.ThT, alpha=0.7)
xx = [random.uniform(.0, 3) for i in d.ThT]
plt.errorbar(x=xx, y=d.ThT, yerr=v.ThT, fmt="o")
texts = []
for i in enumerate(d.ThT):
    texts.append(ax.text(xx[i[0]],i[1],
            '%s'%d.loc[i[0],'comb'].replace(' ','$\n$'),
                          fontsize=4))



plt.ylim([50,7e4])
plt.xlim([-1,4])
plt.axhline(cut,ls='--',color='k')
plt.axhline(901.3333333333334,ls='--')
#adjust_text(texts,arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)

plt.yscale('symlog')
plt.savefig('ThT_gmm_2_point_sem.svg',bbox_inches='tight',transparent=False)
# plt.xscale('log')

In [ ]:
#ThT_96.loc[ThT_96[ThT_96.Seq=='YEVHHQKLVFFA'].index,'Seq']='GYEVHHQKLVFFAED'

In [ ]:
#ThT_96.loc[ThT_96[ThT_96.Seq=='GYEVHHQKLVFFAED'].index,'comb'] = 'GYEVHHQKLVFFAED APP'

In [ ]:
for i in [0,1,2]:
    fig, ax = plt.subplots(figsize=(1.5, 2))
    sns.pointplot(y='ThT',data=ThT_96.loc[ThT_96[ThT_96.Seq.isin(df_membrane.loc[ind][df_membrane.gmm_dSAMs==i].AAseq)].index,:]
                  ,hue='comb',errorbar='se',dodge=True,scale=.51,palette=[colors_gmm[i] for j in range(len(set(ThT_96.loc[ThT_96[ThT_96.Seq.isin(df_membrane.loc[ind][df_membrane.gmm_dSAMs==i].AAseq)].index,:].comb)))])
    plt.yscale('log')
    plt.axhline(cut,ls='--',color='#BDBEC0')
    plt.axhline(901.3333333333334,ls='--',color='#3A3A3B',lw=.5)
    plt.ylim([50,7e4])
    plt.savefig('ThT_gmm_%d_point_sem.svg'%i,bbox_inches='tight',transparent=False)


In [ ]:
for i in [0,1,2]:
    fig, ax = plt.subplots(figsize=(1.5, 3))
    sns.pointplot(y='ThT',data=ThT_96.loc[ThT_96[ThT_96.Seq.isin(df_membrane.loc[ind][df_membrane.gmm_dSAMs==i].AAseq)].index,:]
                  ,hue='comb',errorbar='se',dodge=True,scale=.51,palette='tab20')
    plt.yscale('log')
    plt.axhline(cut,ls='--',color='k')
    plt.axhline(901.3333333333334,ls='--',color='#00ffff')
    plt.legend([])
    plt.savefig('ThT_gmm_%d_point_sem_tab20.svg'%i,bbox_inches='tight',transparent=False)


In [ ]:
for i in enumerate(d.ThT):
    print(i[1])

In [ ]:
GYEVPHQKLVPFAED

In [ ]:
len(dff[(dff.nm==490)&(dff.gmm_new==0)].drop_duplicates('Seq').Seq)

In [ ]:
dff[(dff.well=='C03')&(dff.nm==490)].ThT.mean() 

In [ ]:
fig, ax = plt.subplots(figsize=(3,5))
ord = df[(df.nm==490)&(df.gmm==2)].groupby(['comb']).mean().reset_index().sort_values(by='ThT')['comb']
sns.barplot(y='comb',x='ThT',data=df[(df.nm==490)&(df.gmm==2)],color='#7B8DAE',order=ord,errorbar=('se'),zorder=2,width=.96)
plt.xscale('log')
plt.axvline(df[(df.well=='C03')&(df.nm==490)].ThT.mean(),color='#3C4776',ls='--',lw=2,zorder=0)#+ 
plt.axvline(df[(df.well=='H10')&(df.nm==490)].ThT.mean(),color='#CC6159',ls='--',lw=2,zorder=0)#APP 2M 
plt.xlim([120,5e4])
plt.xticks(fontsize=14)

plt.ylabel('')
plt.xlabel('ThT fluorescence intensity\n(490nm)',fontsize=12)
plt.yticks(fontsize=14)
plt.savefig('plate_ThT_gmm_2.svg',bbox_inches='tight',transparent=False)

In [ ]:
for i in df[(df.gmm.isin(range(3)))].index:
    seq = df.loc[i,'Seq']
    gene_name = df_membrane[df_membrane.AAseq.str.find(seq)!=-1].gene_name.values[0]
    df.loc[i,'gene_name'] = gene_name

In [ ]:
df[df.Seq.str.find('LDH')!=-1]

In [ ]:
df_membrane[df_membrane.AAseq.str.find('LDHCVFTQT')!=-1]

In [ ]:
# Compute the width of each bar. In total we have 2*Pi = 360°
width = 2*np.pi / len(df.index)

# Compute the angle each bar is centered on:
indexes = list(range(1, len(df.index)+1))
angles = [element * width for element in indexes]
angles

# Draw bars
bars = ax.bar(
    x=angles, 
    height=heights, 
    width=width, 
    bottom=lowerLimit,
    linewidth=2, 
    edgecolor="white",
    color="#61a4b2",
)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 8))
ord = df[(df.nm==490)].groupby(['comb']).mean().reset_index().sort_values(by='ThT')['comb']
sns.barplot(y='comb',x='ThT',data=df[(df.nm==490)],color='#7B8DAE',order=ord,errorbar=('se'),zorder=2
            ,hue='gmm',palette=['#A47499','#C56C66','#6C8CBB'],width=.96)
plt.xscale('log')
plt.xscale('log')
plt.yticks(rotation=0)
plt.tight_layout()
plt.axvline(df[(df.well=='H12')&(df.nm==490)].ThT.mean(),color='k',ls='--',lw=.5)#APP 2M 
#ax.grid(zorder=0)
# genes = []
# for seq in enumerate(ord):
#     gene_name = df[df.Seq==seq[1]].gene_name.values[0]
#     ax.text(300,(seq[0]/1)+0.2,'%s'%gene_name,fontsize=16)
#     genes.append(gene_name)
#ax2= ax.twiny()
plt.xticks(fontsize=16)
plt.yticks(fontsize=12)
plt.ylabel('')
plt.xlabel('ThT fluorescence intensity\n(490nm)',fontsize=12)

plt.savefig('plate_ThT_gmm.svg',bbox_inches='tight',transparent=False)

In [ ]:
o = plate_pos2seq_df[plate_pos2seq_df.well.isin(set(df[(df.Seq.str.find('VNATYVNVKCV')!=-1)&(df.nm==490)].well))].odder_number
orederd_peptides[orederd_peptides.order_number.isin(o)]

In [ ]:
plate_pos2seq_df[plate_pos2seq_df.odder_number==21264.0].well.values[0]

In [ ]:
df[(df.Seq.str.find('KCFI')!=-1)&(df.nm==490)]

In [ ]:
#WT and Pro mutations 
t= ['D08','D09','E03','E04','E08','E09','E01','E02','F03','F04','F01','F02','D12','H05','B02','E12']

In [ ]:
#t= set(df[(df.Seq.str.find('P')!=-1)&(df.nm==490)].well)
df[(df.Seq.str.find('P')!=-1)&(~df.well.isin(set(t)))&(df.nm==490)].drop_duplicates('well')

In [ ]:
set(df.Seq)

In [ ]:
pairs =[['KCFIIENIS','KCFIPENIS'],
['CNLVYLIPH','CNLVPLIPH'],
['VSVWVLQS','VSPWVLQS'],
['VLCQYCGLLQI','VLCQYPGLLQI'],
['RVCFWRL','RVCPWRL'],
['DMVEVIYINRK','DMVEPPYINRK'],
['QTLTFCWE','QTLPFCWE'],['YEVHHQKLVFFA','GYEVPHQKLVPFAED']]
# sns.lineplot(x='nm',y='ThT',data=df[df.Seq==pairs[0][0]],errorbar=('se'),err_style='band',color='k')
# sns.lineplot(x='nm',y='ThT',data=df[df.Seq==pairs[0][1]],errorbar=('se'),err_style='band',color='r')
fig = plt.figure(figsize = (8,8),constrained_layout=False)
axs = fig.subplots(nrows=3, ncols=3,gridspec_kw = {'hspace':.42,'wspace':.42})
for j in enumerate(pairs):
    sns.lineplot(x='nm',y='ThT',data=df[df.Seq==j[1][0]],errorbar=('se'),err_style='band',color='k',ax=axs.flat[j[0]])
    sns.lineplot(x='nm',y='ThT',data=df[df.Seq==j[1][1]],errorbar=('se'),err_style='band',color='r',ax=axs.flat[j[0]])
    sns.lineplot(x='nm',y='ThT',data=df[df.well=='H12'],errorbar=('se'),err_style='band',color='#BDBDBF',ax=axs.flat[j[0]])
    axs.flat[j[0]].set_title('%s\n%s/%s'%(df[df.Seq==j[1][0]].gene_name.values[0],j[1][0],j[1][1]),fontsize=10)
    #axs.flat[j[0]].set_ylim([50,3e4])
    #axs.flat[j[0]].set_yscale('log')
    if j[0]==6:
        print('A')
        axs.flat[6].set_ylabel('ThT fluorescence\nintensity')
        axs.flat[6].set_xlabel('nm')
    else:
        axs.flat[j[0]].set_xticklabels([])
        axs.flat[j[0]].set_xlabel('')
        axs.flat[j[0]].set_ylabel('')
        axs.flat[j[0]].set_ylabel('')
    axs.flat[j[0]].grid()
    axs.flat[j[0]].locator_params(axis='x', nbins=4)
    #axs.flat[j[0]].axvline(490,c='k',lw=1,ls='--')
plt.savefig('plate_ThT_PRO_muts.svg',bbox_inches='tight',transparent=False)

In [ ]:
fig = plt.figure(figsize = (10,10),constrained_layout=False)
axs = fig.subplots(nrows=5, ncols=5,gridspec_kw = {'hspace':.42,'wspace':.42})
names = ThT_96.loc[ThT_96[ThT_96.Seq.isin(df_membrane.loc[ind][df_membrane.gmm_dSAMs==0].AAseq)].index,['comb','ThT']].groupby(['comb']).mean().reset_index().sort_values(by='ThT',ascending=0).reset_index()
c = 0
cc = 0
for j in names.index:
    
    sns.lineplot(x='nm',y='ThT',data=df[df.Seq==df[df.comb==names.loc[j,'comb']].Seq.values[0]],errorbar=('se'),err_style='band',color=colors_gmm[0],ax=axs[cc][c])
    sns.lineplot(x='nm',y='ThT',data=df[df.well=='H12'],errorbar=('se'),err_style='band',color='#BDBDBF',ax=axs[cc][c])
    axs[cc][c].set_yscale('log')
    axs[cc][c].set_ylim([50,7e4])
    axs[cc][c].set_ylabel('')
    axs[cc][c].set_title('%s'%names.loc[j,'comb'],fontsize=7)
    c = c+1 
    if c % 5 ==0:
        c = 0
        cc = cc +1 
    if c!=0:
        print('AA')
        axs[cc][c].set_xticklabels([])
        axs[cc][c].set_xlabel('')
        axs[cc][c].set_yticklabels([])
# plt.title('GSVDIVAMD EPOR')
plt.savefig('/Users/aelazar/notebooks_for_paper/ThT_supp/cluster_0.svg',bbox_inches='tight',transparent=False)

In [ ]:
df[df]

In [ ]:
fig = plt.figure(figsize = (21,10),constrained_layout=False)
axs = fig.subplots(nrows=6, ncols=9,gridspec_kw = {'hspace':.45,'wspace':.45})
sns.set(font_scale=1,style='white')
for i in enumerate(set.difference(set(df[(df.nm==490)&(~pd.isna(df.gene_name))].well),set(['H07','H06']))):
    sns.lineplot(x='nm',y='ThT',data=df[df.well==i[1]],errorbar=('se'),err_style='band',ax=axs.flat[i[0]],color='#999999')
    sns.lineplot(x='nm',y='ThT',data=df[df.well=='H12'],errorbar=('se'),err_style='band',color='#BDBDBF',ax=axs.flat[i[0]])
    axs.flat[i[0]].axvline(490,c='k',lw=1,ls='--')
    if i[0]==45:
        axs.flat[45].set_ylabel('ThT fluorescence\nintensity',fontsize=16)
        axs.flat[45].set_xlabel('nm',fontsize=16)

        
    else:
        axs.flat[i[0]].set_xticklabels([])
        axs.flat[i[0]].set_xlabel('')
        axs.flat[i[0]].set_ylabel('')
        axs.flat[i[0]].set_ylabel('')
    axs.flat[i[0]].set_title('%s\n%s'%(df[df.well==i[1]].gene_name.values[0],df[df.well==i[1]].Seq.values[0]))
    #axs.flat[i[0]].ticklabel_format(axis='y', style='scientific',scilimits=(0,0))
    
    axs.flat[i[0]].grid()
    axs.flat[i[0]].locator_params(axis='x', nbins=4)
    plt.savefig('plate_ThT_spectra.svg',bbox_inches='tight',transparent=False)

In [ ]:
fig = plt.figure(figsize = (21,10),constrained_layout=False)
axs = fig.subplots(nrows=6, ncols=9,gridspec_kw = {'hspace':.45,'wspace':.45})
sns.set(font_scale=1,style='white')
for i in enumerate(set.difference(set(df[(df.nm==490)&(~pd.isna(df.gene_name))].well),set(['H07','H06']))):
    sns.lineplot(x='nm',y='ThT',data=df[df.well==i[1]],errorbar=('se'),err_style='band',ax=axs.flat[i[0]],color='k')
    sns.lineplot(x='nm',y='ThT',data=df[df.well=='H12'],errorbar=('se'),err_style='band',color='#317EC2',ax=axs.flat[i[0]])
    axs.flat[i[0]].axvline(490,c='k',lw=1,ls='--')
    if i[0]==45:
        axs.flat[45].set_ylabel('ThT fluorescence\nintensity',fontsize=16)
        axs.flat[45].set_xlabel('nm',fontsize=16)

        
    else:
        axs.flat[i[0]].set_xticklabels([])
        axs.flat[i[0]].set_xlabel('')
        axs.flat[i[0]].set_ylabel('')
        axs.flat[i[0]].set_ylabel('')
    axs.flat[i[0]].set_title('%s\n%s'%(df[df.well==i[1]].gene_name.values[0],df[df.well==i[1]].Seq.values[0]))
    #axs.flat[i[0]].ticklabel_format(axis='y', style='scientific',scilimits=(0,0))
    
    axs.flat[i[0]].grid()
    axs.flat[i[0]].locator_params(axis='x', nbins=4)
    plt.savefig('plate_ThT_spectra.svg',bbox_inches='tight',transparent=False)

In [ ]:
def ThTMAX(seq='LATVQV'):
    dff = df[(df.Seq==seq)].groupby(['Seq','nm']).mean().reset_index()
    ind = dff['ThT'].idxmax()
    return dff.loc[ind]['ThT'],dff.loc[ind]['nm']

In [ ]:
wells = list(set(df[~pd.isna(df.gmm)&(df.nm==490)&(df.ThT>500)].well))

In [ ]:
for ind in df[df.well.isin(wells)].index:
    seq  = df.loc[ind,'Seq']
    
    max_tht ,max_nm = ThTMAX(seq=seq)
    df.loc[ind,'max_tht'] = max_tht
    df.loc[ind,'max_nm'] = max_nm
    if df_membrane[df_membrane.AAseq.str.find(seq)!=-1].empty ==False:
        df.loc[ind,'hydropathy'] = df_membrane[df_membrane.AAseq.str.find(seq)!=-1].hydropathy.values[0]

In [ ]:
sns.scatterplot(x='hydropathy',y='max_nm',data=df[(df.nm==490)&(~pd.isna(df.gmm))],hue='gmm',palette='Dark2')
#plt.xscale('log')

In [ ]:
fig, ax = plt.subplots(figsize=(3, 6))

sns.barplot(x='hydropathy',y='AAseq',data=df_membrane.loc[ind].sort_values(by='hydropathy'),
              palette='Dark2')#native_scale=True,linestyle='none'
plt.yticks(fontsize=8)
plt.tight_layout()
plt.xscale('log')
plt.savefig('aaa.svg',bbox_inches='tight',transparent=True)

In [ ]:
df_membrane[df_membrane.AAseq.str.find('AGVVTTVMGS')!=-1]

In [ ]:
df[df.Seq.str.find('GVVTTVMG')!=-1]#AGVVTTVMGS

In [ ]:
0,3

In [ ]:
df_membrane[df_membrane.gene_name=='TNFRSF13C']